In [0]:
df_sales_info = spark.read.table("db_project.silver.crm_sls_details")
df_customer_info = spark.read.table("db_project.gold.dim_customer_info")
df_product_info = spark.read.table("db_project.gold.dim_product_info")
df_sales_info.limit(1).display()
df_customer_info.limit(1).display()
df_product_info.limit(1).display()

In [0]:
outcome = spark.sql("""
    SELECT 
    sd.sls_ord_num AS Sales_key, 
    ci.Customer_key, 
    pi.Poduct_key, 
    sd.sls_price AS Price,
    sd.sls_quantity AS Quantity, 
    sd.sls_sales AS Sales, 
    sd.sls_order_dt AS Order_date, 
    sd.sls_ship_dt AS Ship_date, 
    sd.sls_due_dt AS Due_date 
    FROM db_project.silver.crm_sls_details sd
    LEFT JOIN db_project.gold.dim_customer_info ci
    ON sd.sls_cust_id = ci.Customer_id
    LEFT JOIN db_project.gold.dim_product_info pi
    ON sd.sls_prd_key =  pi.Subcategory_id
    SORT BY Sales_key
""")
outcome.limit(1).display()

In [0]:
test_nulls = outcome.where(
    outcome.Sales_key.isNull() |
    outcome.Customer_key.isNull() |
    outcome.Poduct_key.isNull() |
    outcome.Price.isNull() |
    outcome.Quantity.isNull() |
    outcome.Sales.isNull() |
    outcome.Order_date.isNull() |
    outcome.Ship_date.isNull() |
    outcome.Due_date.isNull()
)
test_nulls.display()

In [0]:
test_values = outcome.where(
    (outcome.Customer_key < 0) |
    (outcome.Poduct_key < 0) |
    (outcome.Price < 0) |
    (outcome.Quantity < 0) |
    (outcome.Sales < 0)
)
test_values.display()

In [0]:
outcome.write.mode("overwrite").format("delta").saveAsTable("db_project.gold.fact_sales_info")